### 加载数据

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")

In [2]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)
# Inf值检查
inf_cols = X.columns[np.isinf(X).any(axis=0)]
print(inf_cols.tolist())
# 异常收益检查
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

[]
数据异常：收益率超过±30%的列已剔除 3 列 -> ['161811.SZ', '510030.SH', '511580.SH']


### WalkForward + SyntheticData

In [12]:
from skfolio import Population
from sklearn.pipeline import Pipeline
from skfolio.distribution import VineCopula
from skfolio.model_selection import WalkForward
from skfolio.optimization import EqualWeighted
from skfolio.portfolio import MultiPeriodPortfolio
from skfolio.pre_selection import (
    SelectComplete,
    DropZeroVariance,
    SelectNonDominated,
    DropCorrelated,
)
from skfolio.measures import PerfMeasure, RatioMeasure, RiskMeasure

In [14]:
selection_pipe = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated(min_n_assets=5, threshold=-0.4,
                        fitness_measures=[RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN])),
        ("correlate", DropCorrelated(threshold=0.4, absolute=False)),
    ])

In [27]:

# ---------- 合成数据组配置（可自由增删，方便后续比较） ----------
# 每组 = 一个不同的 VineCopula 配置，长度都等于该期测试集
SYNTH_SPECS = {
    "1": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "2": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "3": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "4": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "5": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "6": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "7": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "8": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "9": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "10": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "11": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "12": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    # 压力/条件场景：conditioning 传 {资产名: 值或(下界,上界)}
    # "S3": dict(log_transform=True, random_state=2,
    #            conditioning={"某资产": -0.05}),
}
SYNTH_KEYS = list(SYNTH_SPECS.keys())

# ---------- 容器 ----------
train_portfolios = []
test_portfolios  = MultiPeriodPortfolio()
# 每组合成数据一个 MultiPeriodPortfolio
synth_portfolios = {k: MultiPeriodPortfolio() for k in SYNTH_KEYS}
for k in SYNTH_KEYS:
    synth_portfolios[k].tag = 'SYNTH'

complete_records = []
nondomin_records = []
tailcorr_records = []

cv = WalkForward(test_size=252, train_size=int(252 * 3),
                 purged_size=0, reduce_test=False, expand_train=False)

for i, (train_index, test_index) in enumerate(cv.split(X)):
    # 划分训练测试集
    X_train = X.iloc[train_index]
    X_test  = X.iloc[test_index]
    
    # 筛选
    X_train = selection_pipe.fit_transform(X_train)
    if X_train.empty:
        continue

    # 训练组合
    m = EqualWeighted(portfolio_params=dict(name="Fold %d" % i)).fit(X_train)

    # ① 真实测试集
    train_portfolios.append(m.predict(X_train))
    test_portfolios.append(m.predict(X_test[X_train.columns]))
    #print(X_train.min().min(), X_train.max().max())

    # ② 合成数据测试：每组一个，长度 = 测试集长度
    n_test = len(test_index)
    for g in SYNTH_KEYS:
        spec = SYNTH_SPECS[g]
        # 每期都用当期 X_train 重新拟合 VineCopula，保证资产列对齐
        vine = VineCopula(
            log_transform=spec["log_transform"],
            n_jobs=-1,
            random_state=spec["random_state"],
        )
        # 增加噪声列
        if X_train.shape[1] == 2:
            X_train['noise'] = np.random.uniform(-0.2, 0.2, len(X_train))
        vine.fit(X_train)
        # 关键：vine.sample 返回 (n_test, n_assets) 的 ndarray
        synth_arr = vine.sample(n_samples=n_test, conditioning=spec.get("conditioning"))
        synth_arr = np.clip(synth_arr, -0.2, 0.2)
        #print(synth_arr.min().min(), synth_arr.max().max())
        # 包装成 DataFrame：索引 = X_test.index，列名 = X_train.columns
        synth_test = pd.DataFrame(
            synth_arr, index=X_test.index, columns=X_train.columns
        )
        # 用同一期权重在合成场景上计算组合
        if 'noise' in synth_test.columns:
            synth_test = synth_test.drop(columns=['noise'],axis=0)
        synth_ptf = m.predict(synth_test)
        synth_ptf.name = f"Fold {i} {g}"
        synth_ptf.tag  = g                 # 用 tag 区分合成组
        synth_portfolios[g].append(synth_ptf)

# ---------- 汇总 ----------
population_train = Population(train_portfolios)
population_train.set_portfolio_params(tag="Train")

population_test = Population([test_portfolios])
population_test.set_portfolio_params(tag="Test")

# 统一年化口径（日线=252；周线请改为 52 等）
#population.set_portfolio_params(annualized_factor=252)

print(complete_records)

d:\anaconda3\envs\ml4t\Lib\site-packages\skfolio\distribution\multivariate\_vine_copula.py:629: RuntimeWarning: overflow encountered in exp
  np.exp(samples[:, self._log_transform]) - 1,
d:\anaconda3\envs\ml4t\Lib\site-packages\skfolio\distribution\multivariate\_vine_copula.py:629: RuntimeWarning: overflow encountered in exp
  np.exp(samples[:, self._log_transform]) - 1,
d:\anaconda3\envs\ml4t\Lib\site-packages\skfolio\distribution\multivariate\_vine_copula.py:629: RuntimeWarning: overflow encountered in exp
  np.exp(samples[:, self._log_transform]) - 1,
d:\anaconda3\envs\ml4t\Lib\site-packages\skfolio\distribution\multivariate\_vine_copula.py:629: RuntimeWarning: overflow encountered in exp
  np.exp(samples[:, self._log_transform]) - 1,
d:\anaconda3\envs\ml4t\Lib\site-packages\skfolio\distribution\multivariate\_vine_copula.py:629: RuntimeWarning: overflow encountered in exp
  np.exp(samples[:, self._log_transform]) - 1,
d:\anaconda3\envs\ml4t\Lib\site-packages\skfolio\distribution\mul

[]


In [28]:
Population([synth_portfolios[str(i)] for i in range(1, 13)]+[test_portfolios]).plot_cumulative_returns()

In [29]:
population_test.plot_cumulative_returns()